In [1]:
import pandas as pd
import numpy as np
import re


df = pd.read_csv('dataset_cuaca_bmkg_seluruh_wilayah_indonesia.csv')


missing = df.isna().sum()
laporan_missing = pd.DataFrame({
    'jumlah_missing': missing,
    'persen': (missing / len(df) * 100).round(2)
}).sort_values('persen', ascending=False)
print("--- Laporan Missing Values ---")
print(laporan_missing[laporan_missing['jumlah_missing'] > 0])


print("\n--- Laporan Duplikasi ---")
print("Jumlah baris duplikat:", df.duplicated().sum())


def batas_iqr(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

bawah_suhu, atas_suhu = batas_iqr(df['suhu_c'].dropna())
outlier_suhu = df[(df['suhu_c'] < bawah_suhu) | (df['suhu_c'] > atas_suhu)]
print("\n--- Laporan Outlier ---")
print(f"Jumlah outlier pada Suhu (C): {len(outlier_suhu)}")

--- Laporan Missing Values ---
                  jumlah_missing  persen
jarak_pandang_km            2187   100.0

--- Laporan Duplikasi ---
Jumlah baris duplikat: 0

--- Laporan Outlier ---
Jumlah outlier pada Suhu (C): 19


In [6]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MinMaxScaler
import time
import os


file_name = 'dataset_cuaca_bmkg_seluruh_wilayah_indonesia.csv'
df = pd.read_csv(file_name)
total_baris = len(df)
mem_awal = df.memory_usage(deep=True).sum() / 1024**2

print(f"Total baris data yang diproses: {total_baris:,}")

missing = df.isna().sum()
missing_cols = missing[missing > 0]
print("\n[A] Missing Values:")
if len(missing_cols) == 0:
    print("- Tidak ada missing values (0%).")
else:
    for col, count in missing_cols.items():
        print(f"- {col}: {count} baris ({(count/total_baris)*100:.2f}%)")

dupes = df.duplicated().sum()
print(f"\n[B] Duplikasi Baris Identik:\n- {dupes} baris ({(dupes/total_baris)*100:.2f}%)")

def batas_iqr(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

print("\n[C] Outlier (Nilai Ekstrem):")
outlier_dict = {}
for col in ['suhu_c', 'kelembapan_persen', 'kecepatan_angin_kmjam']:
    bawah, atas = batas_iqr(df[col].dropna())
    outlier_count = ((df[col] < bawah) | (df[col] > atas)).sum()
    outlier_dict[col] = (bawah, atas)
    print(f"- {col}: {outlier_count} baris ({(outlier_count/total_baris)*100:.2f}%)")

inkonsisten = df['desa'].dropna().apply(lambda x: bool(re.search(r'\s{2,}|^[a-z]', str(x)))).sum()
print(f"\n[D] Inkonsistensi Teks (Kolom 'desa'):\n- {inkonsisten} baris ({(inkonsisten/total_baris)*100:.2f}%) butuh dirapikan.")

if 'jarak_pandang_km' in df.columns:
    df = df.drop(columns=['jarak_pandang_km'])

bawah_suhu, atas_suhu = outlier_dict['suhu_c']
df['suhu_c'] = df['suhu_c'].clip(max(bawah_suhu, 0), atas_suhu)
print("[+] Missing values dihapus & Outlier suhu dibatasi (Winsorizing).")


df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
df['local_datetime'] = pd.to_datetime(df['local_datetime'])

for col in ['provinsi', 'kota_kabupaten', 'kecamatan', 'kondisi_cuaca', 'arah_angin']:
    df[col] = df[col].astype('category')

df['suhu_c'] = pd.to_numeric(df['suhu_c'], downcast='integer')
df['kelembapan_persen'] = pd.to_numeric(df['kelembapan_persen'], downcast='integer')
df['kecepatan_angin_kmjam'] = pd.to_numeric(df['kecepatan_angin_kmjam'], downcast='float')
df['tutupan_awan_persen'] = pd.to_numeric(df['tutupan_awan_persen'], downcast='integer')

mem_akhir = df.memory_usage(deep=True).sum() / 1024**2
print("[+] Tipe data dioptimasi (Datetime, Category, Downcasting Integer/Float).")

mm = MinMaxScaler()
df['angin_minmax'] = mm.fit_transform(df[['kecepatan_angin_kmjam']])
df = pd.get_dummies(df, columns=['kondisi_cuaca'], prefix='cuaca', dtype='int8')
print("[+] Normalisasi Min-Max pada angin & One-Hot Encoding pada cuaca diterapkan.")

def bersihkan_teks(teks):
    teks = str(teks).lower()
    teks = re.sub(r'[^a-z\s]', '', teks)
    return re.sub(r'\s+', ' ', teks).strip()

df['desa_bersih'] = df['desa'].fillna('').map(bersihkan_teks)
df = df.drop(columns=['desa'])
print("[+] Pembersihan teks (Regex) pada nama desa selesai.")

nama_file_bersih = 'dataset_cuaca_bmkg_bersih.parquet'
df.to_parquet(nama_file_bersih, index=False)
ukuran_pq = os.path.getsize(nama_file_bersih) / 1024**2
print(f"[+] Data final disimpan ke: {nama_file_bersih} (Ukuran: {ukuran_pq:.2f} MB)")

print(f"Memori Awal (Sebelum optimasi) : {mem_awal:.2f} MB")
print(f"Memori Akhir (Sesudah optimasi): {mem_akhir:.2f} MB")
print(f"Total Penghematan Memori RAM   : {((mem_awal - mem_akhir) / mem_awal) * 100:.1f}%")

Total baris data yang diproses: 53,657

[A] Missing Values:
- jarak_pandang_km: 53657 baris (100.00%)

[B] Duplikasi Baris Identik:
- 0 baris (0.00%)

[C] Outlier (Nilai Ekstrem):
- suhu_c: 548 baris (1.02%)
- kelembapan_persen: 381 baris (0.71%)
- kecepatan_angin_kmjam: 1539 baris (2.87%)

[D] Inkonsistensi Teks (Kolom 'desa'):
- 0 baris (0.00%) butuh dirapikan.
[+] Missing values dihapus & Outlier suhu dibatasi (Winsorizing).
[+] Tipe data dioptimasi (Datetime, Category, Downcasting Integer/Float).
[+] Normalisasi Min-Max pada angin & One-Hot Encoding pada cuaca diterapkan.
[+] Pembersihan teks (Regex) pada nama desa selesai.
[+] Data final disimpan ke: dataset_cuaca_bmkg_bersih.parquet (Ukuran: 0.72 MB)
Memori Awal (Sebelum optimasi) : 30.53 MB
Memori Akhir (Sesudah optimasi): 8.44 MB
Total Penghematan Memori RAM   : 72.3%


In [7]:
import pandas as pd

df_cek = pd.read_parquet('dataset_cuaca_bmkg_bersih.parquet')

df_cek.head()

,kode_wilayah,provinsi,kota_kabupaten,kecamatan,latitude,longitude,datetime_utc,local_datetime,suhu_c,kelembapan_persen,...,angin_minmax,cuaca_Berawan,cuaca_Cerah,cuaca_Cerah Berawan,cuaca_Hujan Petir,cuaca_Hujan Ringan,cuaca_Kabut/Asap,cuaca_Petir,cuaca_Udara Kabur,desa_bersih
0,91.03.05.2001,Papua,Jayapura,Kemtuk,-2.622056,140.346675,2026-10-26 05:00:00+00:00,2026-10-26 14:00:00,30,66,...,0.171617,0,0,0,0,0,0,0,1,sama
1,61.01.06.2001,Kalimantan Barat,Sambas,Sejangkung,1.413380,109.269674,2026-11-12 08:00:00+00:00,2026-11-12 15:00:00,25,94,...,0.145215,0,0,0,0,1,0,0,0,sulung
2,61.02.01.2001,Kalimantan Barat,Mempawah,Mempawah Hilir,0.443870,109.031810,2026-10-24 13:00:00+00:00,2026-10-24 20:00:00,25,101,...,0.171617,0,0,1,0,0,0,0,0,sengkubang
3,33.07.05.2001,Jawa Tengah,Wonosobo,Leksono,-7.433111,109.838483,2026-11-14 23:00:00+00:00,2026-11-15 06:00:00,22,96,...,0.128713,1,0,0,0,0,0,0,0,sawangan
4,91.03.04.2001,Papua,Jayapura,Sentani Barat,-2.505518,140.417044,2026-10-25 22:00:00+00:00,2026-10-26 07:00:00,25,90,...,0.115512,0,0,0,0,0,0,0,1,dosay
